# Gradient Descent

Gradient descent is how most of modern AI is trained: take the gradient of
a loss function, step against it, repeat. This notebook uses the repo's
own `Calculus.gradient_descent_step` and `Calculus.newton_step`
(`src/math_utils/calculus.py`) to minimize a simple 2D function, visualize
the descent path, and compare first-order (gradient) vs. second-order
(Newton) convergence.

## Table of Contents
1. [A test loss surface](#surface)
2. [Gradient descent, step by step](#descent)
3. [Gradient descent vs. Newton's method](#compare)


<a id='surface'></a>
## 1. A test loss surface

A function shaped like a stretched bowl - convex, with a single minimum at `(0, 0)`. This is a stand-in for a loss surface: real training loss landscapes are far messier, but the mechanics of "follow the gradient downhill" are the same.

In [ ]:
import sys
sys.path.insert(0, "../../src")

import numpy as np
import matplotlib.pyplot as plt
from math_utils.calculus import Calculus


def loss(x: np.ndarray) -> float:
    """f(x, y) = x^2 + 5y^2 - a stretched bowl, minimum at the origin."""
    return x[0] ** 2 + 5 * x[1] ** 2


x_range = np.linspace(-4, 4, 200)
y_range = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(x_range, y_range)
Z = X ** 2 + 5 * Y ** 2

fig, ax = plt.subplots(figsize=(7, 5))
contours = ax.contour(X, Y, Z, levels=20, cmap="viridis")
ax.clabel(contours, inline=True, fontsize=8)
ax.set_title("Loss surface: f(x, y) = x^2 + 5y^2")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()


<a id='descent'></a>
## 2. Gradient descent, step by step

Each call to `Calculus.gradient_descent_step` numerically differentiates `loss` and takes one step against the gradient. Chaining calls traces the descent path.

In [ ]:
def run_gradient_descent(f, x0, learning_rate=0.1, n_steps=25):
    path = [np.array(x0, dtype=float)]
    x = np.array(x0, dtype=float)
    for _ in range(n_steps):
        x = Calculus.gradient_descent_step(f, x, learning_rate=learning_rate)
        path.append(x.copy())
    return np.array(path)


path = run_gradient_descent(loss, x0=[3.5, 1.8], learning_rate=0.1, n_steps=25)
print("Start:", path[0], "-> End:", path[-1])
print("Final loss:", loss(path[-1]))

fig, ax = plt.subplots(figsize=(7, 5))
contours = ax.contour(X, Y, Z, levels=20, cmap="viridis", alpha=0.6)
ax.plot(path[:, 0], path[:, 1], "o-", color="crimson", markersize=3, label="gradient descent path")
ax.scatter(*path[0], color="black", zorder=5, label="start")
ax.scatter(0, 0, color="gold", marker="*", s=150, zorder=5, label="minimum")
ax.legend()
ax.set_title(f"Gradient descent: {len(path) - 1} steps, lr=0.1")
plt.show()


<a id='compare'></a>
## 3. Gradient descent vs. Newton's method

Newton's method (`Calculus.newton_step`) uses curvature (the Hessian) as well as the gradient, so it can converge in far fewer steps on a well-behaved convex function like this one - at the cost of computing and inverting a Hessian each step, which is expensive in high dimensions. This is exactly the first-order vs. second-order optimizer trade-off behind SGD vs. Adam vs. L-BFGS in real training loops.

In [ ]:
def run_newton(f, x0, n_steps=10):
    path = [np.array(x0, dtype=float)]
    x = np.array(x0, dtype=float)
    for _ in range(n_steps):
        x = Calculus.newton_step(f, x)
        path.append(x.copy())
    return np.array(path)


newton_path = run_newton(loss, x0=[3.5, 1.8], n_steps=10)

gd_losses = [loss(p) for p in path]
newton_losses = [loss(p) for p in newton_path]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(gd_losses, "o-", label=f"gradient descent ({len(path) - 1} steps)")
ax.plot(newton_losses, "s-", label=f"Newton's method ({len(newton_path) - 1} steps)")
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("loss (log scale)")
ax.set_title("Convergence: gradient descent vs. Newton's method")
ax.legend()
plt.show()

print(f"Gradient descent reached loss={gd_losses[-1]:.2e} in {len(path) - 1} steps")
print(f"Newton's method reached loss={newton_losses[-1]:.2e} in {len(newton_path) - 1} steps")


## Takeaways

- `Calculus.gradient_descent_step(f, x, learning_rate)` takes one
  numerically-differentiated step downhill - this is the core loop inside
  every SGD/Adam/RMSprop optimizer in `docs/` and in real frameworks.
- `Calculus.newton_step(f, x)` uses second-order (Hessian) information to
  converge faster on well-behaved functions, at higher per-step cost -
  the same trade-off L-BFGS makes against plain SGD.
- See also: `scripts/optimization_routines.py` for a runnable script
  version of this comparison, and
  `notebooks/linear-algebra/02_eigendecomposition.ipynb` for how
  eigenvalues explain *why* a stretched bowl like `x^2 + 5y^2` is harder
  for plain gradient descent than a circular one.
